In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"SuperStore_Orders.csv", encoding="latin-1")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        51290 non-null  str    
 1   order_date      51290 non-null  str    
 2   ship_date       51290 non-null  str    
 3   ship_mode       51290 non-null  str    
 4   customer_name   51290 non-null  str    
 5   segment         51290 non-null  str    
 6   state           51290 non-null  str    
 7   country         51290 non-null  str    
 8   market          51290 non-null  str    
 9   region          51290 non-null  str    
 10  product_id      51290 non-null  str    
 11  category        51290 non-null  str    
 12  sub_category    51290 non-null  str    
 13  product_name    51290 non-null  str    
 14  sales           51290 non-null  str    
 15  quantity        51290 non-null  int64  
 16  discount        51290 non-null  float64
 17  profit          51290 non-null  float64
 1

In [3]:
df['ship_date'] = pd.to_datetime(df['ship_date'], format="%d-%m-%Y")
df['order_date'] = pd.to_datetime(df['order_date'], format="%d-%m-%Y")

print(df['order_date'].describe(), df['ship_date'].describe())

count                         51290
mean     2013-05-11 21:26:49.155780
min             2011-01-01 00:00:00
25%             2012-06-19 00:00:00
50%             2013-07-08 00:00:00
75%             2014-05-22 00:00:00
max             2014-12-31 00:00:00
Name: order_date, dtype: object count                         51290
mean     2013-05-15 20:42:42.745174
min             2011-01-03 00:00:00
25%             2012-06-23 00:00:00
50%             2013-07-12 00:00:00
75%             2014-05-26 00:00:00
max             2015-01-07 00:00:00
Name: ship_date, dtype: object


In [4]:
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df['sales'].describe()

count    48660.000000
mean       161.017838
std        201.092519
min          0.000000
25%         29.000000
50%         77.000000
75%        208.000000
max        999.000000
Name: sales, dtype: float64

In [5]:
print("Unique orders:", df['order_id'].nunique(),"\nUnique customers:", df['customer_name'].nunique(),"\nUnique products:", df['product_id'].nunique())

Unique orders: 25035 
Unique customers: 795 
Unique products: 10292


In [6]:
custom_map = {
    name: i + 1
    for i, name in enumerate(df['customer_name'].unique())
}

In [7]:
df['customer_id'] = df['customer_name'].map(custom_map)

In [8]:
print(df[["customer_id", "customer_name"]].drop_duplicates().head(10))

    customer_id      customer_name
0             1    Toby Braunhardt
1             2        Joseph Holt
2             3      Annie Thurman
3             4       Eugene Moren
6             5    Magdelene Morse
7             6        Kean Nguyen
8             7       Ken Lonsdale
9             8   Lindsay Williams
10            9       Larry Blacks
12           10  Dorothy Dickinson


In [9]:
df.groupby("customer_name")["customer_id"].nunique().max()

np.int64(1)

In [10]:
print("Unique orders:", df["order_id"].nunique())
print("Unique customers:", df["customer_name"].nunique())
print("Unique products:", df["product_id"].nunique())
print("Unique customer id's:", df["customer_id"].nunique())

Unique orders: 25035
Unique customers: 795
Unique products: 10292
Unique customer id's: 795


In [11]:
df.groupby("order_id")["customer_name"].nunique().max()

np.int64(4)

In [12]:
df.groupby("product_id")["product_name"].nunique().max()

np.int64(4)

In [13]:
product_check = (df.groupby("order_id")["customer_name"].nunique().sort_values(ascending=False))

In [14]:
print(product_check)

order_id
ES-2013-4477863    4
ES-2014-3566095    4
ES-2014-4717877    4
IT-2014-1036058    4
ES-2014-2939767    3
                  ..
AG-2011-8390       1
AG-2011-8930       1
AG-2011-9060       1
AG-2011-9610       1
AE-2014-4120       1
Name: customer_name, Length: 25035, dtype: int64


In [15]:
problem_products = product_check[product_check > 1].index

print(problem_products)

Index(['ES-2013-4477863', 'ES-2014-3566095', 'ES-2014-4717877',
       'IT-2014-1036058', 'ES-2014-2939767', 'ES-2012-1721838',
       'ES-2013-2003988', 'ES-2012-5064111', 'ES-2014-2275791',
       'ES-2013-3845157',
       ...
       'US-2014-141558', 'US-2014-144582', 'US-2014-146213', 'US-2014-147221',
       'US-2014-152492', 'US-2014-155425', 'US-2014-161830', 'US-2014-166394',
       'UZ-2014-6400', 'ZI-2014-1780'],
      dtype='str', name='order_id', length=659)


In [16]:
product_check = (
    df.groupby("product_id")["product_name"]
      .nunique()
      .sort_values(ascending=False)
)

print(product_check.head(10))

product_id
OFF-PA-10004673    4
OFF-BI-10004632    3
FUR-FU-10004960    3
OFF-BI-10003708    3
OFF-EN-10003737    3
OFF-BI-10002799    3
FUR-CH-10002335    3
FUR-FU-10001933    3
OFF-AR-10000833    3
OFF-BI-10004654    3
Name: product_name, dtype: int64


In [17]:
problem_products = product_check[product_check > 1].index

print("Number of problematic products:", len(problem_products))
print(problem_products[:20])

Number of problematic products: 457
Index(['OFF-PA-10004673', 'OFF-BI-10004632', 'FUR-FU-10004960',
       'OFF-BI-10003708', 'OFF-EN-10003737', 'OFF-BI-10002799',
       'FUR-CH-10002335', 'FUR-FU-10001933', 'OFF-AR-10000833',
       'OFF-BI-10004654', 'FUR-BO-10002204', 'OFF-AR-10003829',
       'OFF-AR-10003651', 'OFF-PA-10000357', 'OFF-SU-10003629',
       'TEC-AC-10002842', 'TEC-AC-10003063', 'TEC-AC-10004334',
       'OFF-AR-10000390', 'OFF-AR-10000399'],
      dtype='str', name='product_id')


In [18]:
df[df["product_id"] == "OFF-PA-10004673"][
    ["product_id", "product_name", "category", "sub_category"]
].drop_duplicates()

,product_id,product_name,category,sub_category
9209,OFF-PA-10004673,"Enermax Message Books, 8.5 x 11",Office Supplies,Paper
16066,OFF-PA-10004673,"Eaton Note Cards, Multicolor",Office Supplies,Paper
38252,OFF-PA-10004673,"Xerox Message Books, Multicolor",Office Supplies,Paper
45651,OFF-PA-10004673,"Green Bar Memo Slips, Recycled",Office Supplies,Paper


In [19]:
print(
    df[["product_id", "product_name"]]
    .drop_duplicates()
    .shape
)

(10768, 2)


In [20]:
print(df["product_id"].nunique())

10292


In [21]:
print(
    df.groupby("product_id")["product_name"]
      .nunique()
      .value_counts()
      .sort_index()
)

product_name
1    9835
2     439
3      17
4       1
Name: count, dtype: int64


In [22]:
df[df["order_id"] == "ES-2013-4477863"][
    [
        "order_id",
        "customer_name",
        "product_id",
        "product_name",
        "sales",
        "quantity",
        "profit"
    ]
]

,order_id,customer_name,product_id,product_name,sales,quantity,profit
27364,ES-2013-4477863,Dave Hallsten,OFF-BI-10003705,"Wilson Jones Binding Machine, Recycled",247.0,5,27.000
29020,ES-2013-4477863,Maxwell Schwartz,OFF-BI-10003642,"Wilson Jones Binder, Economy",13.0,1,4.020
30303,ES-2013-4477863,Marc Crier,OFF-AR-10000799,"Sanford Highlighters, Easy-Erase",32.0,2,1.260
31312,ES-2013-4477863,Jason Gross,OFF-BI-10003650,"Ibico Index Tab, Clear",4.0,1,-2.865


In [23]:
customers = df[["customer_id", "customer_name", "segment", "country", "state"]].drop_duplicates("customer_id")

In [24]:
print(customers.shape)
print(customers.head())

(795, 5)
   customer_id    customer_name      segment    country            state
0            1  Toby Braunhardt     Consumer    Algeria      Constantine
1            2      Joseph Holt     Consumer  Australia  New South Wales
2            3    Annie Thurman     Consumer    Hungary         Budapest
3            4     Eugene Moren  Home Office     Sweden        Stockholm
6            5  Magdelene Morse     Consumer     Canada          Ontario


In [25]:
print(customers["customer_id"].nunique())

795


In [26]:
products = df[["product_id", "product_name", "category", "sub_category"]].drop_duplicates()

In [27]:
products.insert(0, "product_key", range(1, len(products) + 1))

In [28]:
print(products.shape)
print(products.head())

(10768, 5)
   product_key        product_id                 product_name  \
0            1  OFF-TEN-10000025          Tenex Lockers, Blue   
1            2   OFF-SU-10000618     Acme Trimmer, High Speed   
2            3  OFF-TEN-10001585      Tenex Box, Single Width   
3            4   OFF-PA-10001492  Enermax Note Cards, Premium   
4            5   FUR-FU-10003447   Eldon Light Bulb, Duo Pack   

          category sub_category  
0  Office Supplies      Storage  
1  Office Supplies     Supplies  
2  Office Supplies      Storage  
3  Office Supplies        Paper  
4        Furniture  Furnishings  


In [29]:
sales = df.merge(
    products[["product_key", "product_id", "product_name"]],
    on=["product_id", "product_name"],
    how="left"
)

In [30]:
sales.shape

(51290, 23)

In [31]:
sales['product_key'].isnull().sum()

np.int64(0)

In [32]:
sales = sales[
    [
        "order_id",
        "customer_id",
        "product_key",
        "order_date",
        "ship_date",
        "ship_mode",
        "market",
        "region",
        "quantity",
        "sales",
        "discount",
        "profit",
        "shipping_cost",
        "order_priority",
        "year"
    ]
].copy()

In [33]:
sales.insert(
    0,
    "sales_record_id",
    range(1, len(sales) + 1)
)

In [34]:
sales.shape

(51290, 16)

In [35]:
sales.head()

,sales_record_id,order_id,customer_id,product_key,order_date,ship_date,ship_mode,market,region,quantity,sales,discount,profit,shipping_cost,order_priority,year
0,1,AG-2011-2040,1,1,2011-01-01,2011-01-06,Standard Class,Africa,Africa,2,408.0,0.0,106.140,35.46,Medium,2011
1,2,IN-2011-47883,2,2,2011-01-01,2011-01-08,Standard Class,APAC,Oceania,3,120.0,0.1,36.036,9.72,Medium,2011
2,3,HU-2011-1220,3,3,2011-01-01,2011-01-05,Second Class,EMEA,EMEA,4,66.0,0.0,29.640,8.17,High,2011
3,4,IT-2011-3647632,4,4,2011-01-01,2011-01-05,Second Class,EU,North,3,45.0,0.5,-26.055,4.82,High,2011
4,5,IN-2011-47883,2,5,2011-01-01,2011-01-08,Standard Class,APAC,Oceania,5,114.0,0.1,37.770,4.70,Medium,2011


In [36]:
print("Customers:", len(customers))
print("Products:", len(products))
print("Sales:", len(sales))

print("\nCustomer IDs unique:",
      customers["customer_id"].is_unique)

print("Product keys unique:",
      products["product_key"].is_unique)

print("Sales IDs unique:",
      sales["sales_record_id"].is_unique)

print("\nMissing customer IDs:",
      sales["customer_id"].isnull().sum())

print("Missing product keys:",
      sales["product_key"].isnull().sum())

Customers: 795
Products: 10768
Sales: 51290

Customer IDs unique: True
Product keys unique: True
Sales IDs unique: True

Missing customer IDs: 0
Missing product keys: 0


In [38]:
customers.to_csv(r"C:\Users\HP\OneDrive\Desktop\Sem-7\Major Project\Processed Datasets\customers.csv", index=False)
products.to_csv(r"C:\Users\HP\OneDrive\Desktop\Sem-7\Major Project\Processed Datasets\products.csv", index=False)
sales.to_csv(r"C:\Users\HP\OneDrive\Desktop\Sem-7\Major Project\Processed Datasets\sales.csv", index=False)